In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'Adresse': ['1 rue de la Paix', '2 av. des Champs', '3 bd. Voltaire'],
    'code_postal_brut': [75013, 75020, 75013],
    'Surface': [100, 150, 80]
})
print("1. DataFrame 'df' initial :")
print(df)
print("-" * 30)

In [ ]:
import pandas as pd
import numpy as np

df_base = pd.DataFrame(
    {
        "ID": [1, 2, 3, 4, 5],
        "Nom": ["Alice", "Bob", "Charlie", "David", "Eve"],
        "Age": [25, 30, 22, 35, 28],
        "Ville": ["Paris", "Lyon", "Paris", "Marseille", "Lyon"],
        "Statut_Logement": [
            "Social",
            "Privé",
            "Social",
            "Privé",
            "Social",
        ],  # Nouveau : type de logement
        "Is_Renovated": [1, 0, 1, 1, 0],  # Nouveau : 1 si rénové, 0 sinon
    }
)
# Remplace les noms de colonnes pour qu'ils ressemblent plus à ton cas :
df_base.rename(
    columns={"Ville": "Code_Postal_Brut", "Statut_Logement": "Statut_Juridique"},
    inplace=True,
)
df_base

In [ ]:

renovation_grouped = df_base.groupby(['Code_Postal_Brut', 'Statut_Juridique'])
# print('renovation_grouped: ', renovation_grouped)
renovation_grouped


In [ ]:
res=renovation_grouped['Age'].mean()
res


In [ ]:
result=res.unstack()
result


In [ ]:
columns=res.unstack().columns
columns

In [ ]:
MultiIndex=pd.MultiIndex.from_product([['mean'], ('mean',  'Privé')])
MultiIndex

In [ ]:
# res.unstack().columns=pd.MultiIndex.from_product([['mean'], res.unstack().columns])
new_MultiIndex_columns=res.unstack().columns=pd.MultiIndex.from_product([['mean'], ('mean',  'Privé')])
new_MultiIndex_columns


In [ ]:
index_columns_sociale=columns.get_level_values(0)
index_columns_sociale

In [ ]:
index_columns_privet=columns.get_level_values(-1)
index_columns_privet

In [ ]:


print('index_columns_sociale: ', index_columns_sociale)
print('index_columns_sociale: ', index_columns_privet)

In [ ]:
import pandas as pd
from typing import List, Optional
from pathlib import Path
import logging
import os


class Data_recontracter:

    def __init__(
        self,
        data_light_source_path="./data_light_output/dpe03existant_paris_light.csv",
        columns_df_importent: Optional[List[str]] = None,
    ):

        self.columns_df_importent = columns_df_importent
        self.data_light_source_path: pd.DataFrame = data_light_source_path
        self.df_light: pd.DataFrame = None

    def create_light_df_source(self, file_name: str = None) -> str:
        file_path = None
        if file_name is not None:
            file_path = self._get_file_path(file_name)
            if not file_path:
                print("file_name Se trouve pas dans le répertoire /data_source_inputs")
                return file_path

        self.df_light = self._load_clean_csv(file_path)
        output_path = f"{self.output_dir}/{file_name}_light.csv"
        if os.path.exists(output_path):
            print(f" Le fichier {fi} existe")
            return output_path

        self.df_light.to_csv(output_path, index=False)
        print("output_path: ", output_path)
        return output_path

    # privet method
    def _add_computed_columns(self) -> Path:
        new_df = None

        self.df_light["statut_juridique"] = "Sociale"

        condition_sociale = self.df_light["numero_rpls_logement"].notna()
        condition_privet = self.df_light["numero_immatriculation_copropriete"].notna()

        self.df_light.loc[condition_sociale, "statut_juridique"] = "Sociale"
        self.df_light.loc[condition_privet, "statut_juridique"] = "Privet"

        is_renovated = self.df_light["etiquette_dpe"].isin(["A", "B", "C", "D"])
        self.df_light["is_renovated"] = is_renovated.astype(int)

        print("is_renovated: ", is_renovated)
        # cp_group
        renovation_stats_by_cp_group = self.df_light.groupby("code_postal_brut")

        renovation_stats_by_cp_group["is_renovated"].agg(
            {"moyenne": "mean", "total": "sum"}
        )

        renovation_stats_by_cp_group["pourcentage_renovated"] = (
            renovation_stats_by_cp_group["moyenne"] * 100
        )

        renovation_stats_by_cp_group["pourcentage_renovated"].round(1)
        print("renovation_stats_by_cp_group: ", renovation_stats_by_cp_group)

        # cp_status_group
        renovation_stats_by_cp_status_group = self.df_light.groupby(
            ["code_postal_brut", "statut_juridique"]
        )

        renovation_stats_series = renovation_stats_by_cp_status_group[
            "is_renovated"
        ].agg({"moyenne": "mean"})

        renovation_stats_series["pourcentage_renovated"] = (
            renovation_stats_series["moyenne"] * 100
        ).round(1)

    df["code_postal_brut"].map(
        renovation_pivot.get_level_values(1).map(
            {"Social": renovation_pivot.loc[:, ("mean", "Social")] * 100}
        )
        if ("mean", "Social") in renovation_pivot.columns
        else 0
    )


if __name__ == "__main__":
    d1 = Data_loader("./data_light_output")
    pathName = d1.create_light_df_source("dpe03existant_paris")
    print("pathName: ", pathName)